# Research Paper Analyst

An AI-powered research agent that uses the **Plan-and-Execute** pattern to analyze any research topic.

Given a query, the agent:
1. Searches the web for relevant research papers and articles
2. Formulates a dynamic research plan (3-5 steps)
3. Executes each step against the retrieved content
4. Compiles a comprehensive Markdown report

**Tech stack**: LangGraph, LangChain, Tavily Search, OpenAI GPT-4o-mini

In [ ]:
!pip install langgraph langchain-openai langchain-core pydantic python-dotenv tavily-python -q

## Imports and Configuration

In [ ]:
from __future__ import annotations
import os
import textwrap
import operator
from typing import Annotated, Literal, TypedDict, Any

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from tavily import TavilyClient
from IPython.display import Markdown, display

In [ ]:
from dotenv import load_dotenv
import os

# Load API keys from .env file
load_dotenv()

# Check if keys are loaded
if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("TAVILY_API_KEY"):
    print("⚠️ Warning: API keys not found in .env file!")
else:
    print("API keys loaded successfully ✅")

# model and search settings
PLANNER_CONTEXT_CHARS = 4000
DEFAULT_MODEL = "gpt-4o-mini"
TAVILY_MAX_RESULTS = 5

## State Definition

This `TypedDict` is the shared memory that flows through every node in the graph. Each node reads what it needs and writes back its results.

We use `Annotated` with `operator.add` for list fields so LangGraph knows how to merge updates correctly.

In [ ]:
class AgentState(TypedDict):
    research_objective: str       # the user's query
    raw_text: str                 # full text from search results
    sources_info: str             # formatted string of source URLs
    plan: list[str]               # the generated research plan
    current_step_index: int       # which step we're on
    step_results: Annotated[list[str], operator.add]  # findings from each step
    final_report: str             # the compiled report

## Pydantic Schema for Plan Output

We use structured output so the LLM is forced to return exactly 3-5 steps as a clean list — no parsing needed on our end.

In [ ]:
class ResearchPlan(BaseModel):
    steps: list[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3-5 specific, actionable research steps.",
    )

## LLM Helper

A small wrapper so we can swap models easily without touching every node.

In [ ]:
def get_llm(temperature=0.0):
    return ChatOpenAI(
        model=DEFAULT_MODEL,
        temperature=temperature,
        api_key=os.environ["OPENAI_API_KEY"],
    )

## Search Node (Tavily)

Uses Tavily's advanced search to pull relevant articles and papers from the web. The results get concatenated into `raw_text` which later nodes will analyze.

In [ ]:
def search_node(state: AgentState) -> dict:
    query = state["research_objective"]
    print(f"\n{'='*60}")
    print(f"[SEARCH] Searching the web for: {query}")
    print(f"{'='*60}")

    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=TAVILY_MAX_RESULTS,
        include_answer=True,
        topic="general",
    )

    text_chunks = []
    sources_list = []

    # tavily sometimes returns an AI-generated overview
    if response.get("answer"):
        text_chunks.append(f"=== AI OVERVIEW ===\n{response['answer']}\n")

    for i, r in enumerate(response.get("results", []), 1):
        title = r.get("title", "Untitled")
        url = r.get("url", "")
        content = r.get("content", "")

        text_chunks.append(f"=== SOURCE {i}: {title} ===\nURL: {url}\n\n{content}\n")
        sources_list.append(f"[{i}] {title} - {url}")
        print(f"  [{i}] {title}")

    raw_text = "\n\n".join(text_chunks)
    sources_info = "\n".join(sources_list)

    print(f"\n[SEARCH] Done - {len(sources_list)} sources, {len(raw_text):,} chars total")

    return {"raw_text": raw_text, "sources_info": sources_info}

## Planner Node

This is where the "Plan" in Plan-and-Execute happens. The LLM reads the research objective + a preview of the content, then generates 3-5 concrete steps to investigate.

In [ ]:
def planner_node(state: AgentState) -> dict:
    print(f"\n{'='*60}")
    print(f"[PLANNER] Generating research plan...")
    print(f"{'='*60}")

    llm = get_llm(temperature=0.2)
    structured_llm = llm.with_structured_output(ResearchPlan)

    system_msg = """You are an expert research analyst. Create a focused, tactical
research plan for analyzing web search results on a topic.

Rules:
- Generate exactly 3 to 5 specific, actionable steps.
- Each step should be a clear question or extraction task that can be answered from the content.
- Order them logically (background before findings, methodology before results).
- Don't include generic steps like 'read the content'. Be specific to the topic."""

    human_msg = f"""OBJECTIVE: {state['research_objective']}

CONTENT PREVIEW:
---
{state['raw_text'][:PLANNER_CONTEXT_CHARS]}
---

Generate 3-5 tactical research steps."""

    result = structured_llm.invoke([
        SystemMessage(content=system_msg),
        HumanMessage(content=human_msg),
    ])

    print(f"\nPlan ({len(result.steps)} steps):")
    for i, s in enumerate(result.steps, 1):
        print(f"  {i}. {s}")

    return {
        "plan": result.steps,
        "current_step_index": 0,
    }

## Executor Node

The "Execute" part — this runs one plan step at a time. It sends the current step + the full content to the LLM, extracts the relevant findings, and bumps the step counter.

In [ ]:
def executor_node(state: AgentState) -> dict:
    idx = state["current_step_index"]
    step = state["plan"][idx]
    total = len(state["plan"])

    print(f"\n[EXECUTOR] Step {idx+1}/{total}: {step}")

    llm = get_llm(temperature=0.0)

    system_msg = """You are a meticulous research analyst. You're given content from
research sources and one specific task to complete.

Instructions:
- Focus ONLY on answering the given task.
- Pull out concrete data points, quotes, methodology details, or findings.
- If the info isn't available in the content, say so clearly. Don't make things up.
- Reference sources by name/title when you cite them.
- Keep it thorough but concise — about 150-400 words. Bullet points are fine."""

    human_msg = f"""TASK:\n{step}\n\nCONTENT:\n---\n{state['raw_text']}\n---\n\nProvide your detailed findings for this task."""

    response = llm.invoke([
        SystemMessage(content=system_msg),
        HumanMessage(content=human_msg),
    ])

    finding = response.content

    # format as "Step: ... \n Finding: ..." so the synthesizer can parse it
    formatted = f"STEP: {step}\nFINDING:\n{finding}"

    preview = finding[:200].replace('\n', ' ').strip()
    print(f"  -> {preview}...")

    return {
        "step_results": [formatted],       # appended via operator.add
        "current_step_index": idx + 1,
    }

## Step Router

A conditional edge function that checks whether we have more steps to execute. If yes, loop back to the executor. If all steps are done, move to the synthesizer.

In [ ]:
def step_router(state: AgentState) -> Literal["executor_node", "synthesizer_node"]:
    idx = state["current_step_index"]
    total = len(state["plan"])

    if idx < total:
        print(f"[ROUTER] {total - idx} steps left -> back to executor")
        return "executor_node"

    print(f"[ROUTER] All {total} steps done -> moving to synthesizer")
    return "synthesizer_node"

## Synthesizer Node

Takes all the findings from every executed step and compiles them into a clean, structured Markdown report with an executive summary, cross-cutting analysis, and source citations.

In [ ]:
def synthesizer_node(state: AgentState) -> dict:
    print(f"\n{'='*60}")
    print("[SYNTHESIZER] Compiling final report...")
    print(f"{'='*60}")

    llm = get_llm(temperature=0.3)

    # combine all step results into one block
    all_findings = "\n\n".join(state["step_results"])

    plan_text = "\n".join(f"{i}. {s}" for i, s in enumerate(state["plan"], 1))

    system_msg = """You are a senior research analyst writing a final report.

Use this structure:
1. # Research Analysis Report
2. ## Executive Summary (2-3 sentences)
3. ## Detailed Findings (one ### subsection per step)
4. ## Cross-Cutting Analysis (patterns, connections across findings)
5. ## Limitations & Further Research
6. ## Conclusion
7. ## Sources (list all web sources)

Rules:
- Use proper Markdown — headers, bullets, bold, italics, blockquotes.
- Aim for 800-1500 words. Be thorough but don't pad.
- Don't make up information that isn't in the findings.
- Cite web sources with [Source N] notation where relevant."""

    human_msg = f"""OBJECTIVE: {state['research_objective']}

PLAN EXECUTED:
{plan_text}

FINDINGS:
{all_findings}

SOURCES:
{state['sources_info']}

Write the final Markdown report."""

    response = llm.invoke([
        SystemMessage(content=system_msg),
        HumanMessage(content=human_msg),
    ])

    print(f"[SYNTHESIZER] Done - {len(response.content):,} chars")
    return {"final_report": response.content}

## Building the State Graph

Now we wire everything together using LangGraph's `StateGraph`. The architecture:

```
START -> search_node -> planner_node -> executor_node <-> step_router -> synthesizer_node -> END
```

The executor and router form a loop — the executor runs one step, the router checks if there's more, and sends it back if needed.

In [ ]:
# initialize the graph
builder = StateGraph(AgentState)

# add all the nodes
builder.add_node("search_node", search_node)
builder.add_node("planner_node", planner_node)
builder.add_node("executor_node", executor_node)
builder.add_node("synthesizer_node", synthesizer_node)

# wire the edges: search -> plan -> execute -> route -> (loop or synthesize)
builder.add_edge(START, "search_node")
builder.add_edge("search_node", "planner_node")
builder.add_edge("planner_node", "executor_node")

# the executor loop — keeps going until all steps are done
builder.add_conditional_edges("executor_node", step_router, {
    "executor_node": "executor_node",
    "synthesizer_node": "synthesizer_node",
})

# synthesizer is the final stop
builder.add_edge("synthesizer_node", END)

# compile
graph = builder.compile()

print("Graph compiled!")
print(f"Nodes: {list(graph.nodes.keys())}")

## The `analyze()` Function

A convenience wrapper that invokes the graph and renders the report as formatted Markdown right in the notebook.

In [ ]:
def analyze(query):
    """
    Run the research agent on any topic.
    
    Args:
        query: what you want to research (e.g. "latest trends in EV batteries")
    """
    result = graph.invoke({
        "research_objective": query,
        "raw_text": "",
        "sources_info": "",
        "plan": [],
        "current_step_index": 0,
        "step_results": [],
        "final_report": "",
    })

    display(Markdown("---"))
    display(Markdown(result["final_report"]))
    return result["final_report"]

---

## Running the Agent

Just change the query string and run the cell.

In [ ]:
report = analyze("latest trends in EV batteries")

In [ ]:
# try another topic
# report = analyze("advances in transformer architectures 2024")

In [ ]:
# report = analyze("CRISPR gene editing breakthroughs")

In [ ]:
# report = analyze("impact of LLMs on software engineering")